# 05｜求導法則與連鎖律：讓電腦幫你驗算

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wang3yuan/calculus-for-iih/blob/main/notebooks/05_derivative_rules.ipynb)

臺北醫學大學 智慧醫療跨領域　微積分（1151）　**自由練習，不計分、不用繳交**

對應課堂投影片：**Calculus05 求導法則與連鎖律**

**怎麼用**
1. 按上方選單「執行階段 → 全部執行」，從上往下看圖和數字。
2. 看到 **✏️ 動手** 的格子，就自己改改看。填空沒填也不會出錯，程式會提醒你「還沒填」。
3. 卡住就點開「💡 提示」，真的不行再看「參考答案」。

不需要先會寫 Python，每一段程式前面都有中文說明。

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

def d(f, a, h=1e-6):
    "中央差分：用前後兩點估計導數（老師提供）。"
    return (f(a + h) - f(a - h)) / (2 * h)

def check(ok, hint):
    """檢查答案：對了顯示 ✓，不對顯示提示。"""
    print("✓ 正確！" if ok else "✗ 還沒對，" + hint)

## 1. 認識 SymPy：會做代數的 Python

`sympy` 可以直接算導函數。下面看冪次法則 $(x^n)'=nx^{n-1}$。

In [ ]:
x = sp.symbols("x")
for p in [2, 3, 5, sp.Rational(1, 2), -1]:
    print(f"(x^{p})′ = {sp.diff(x**p, x)}")

## 2. 陷阱：乘積的導數不是導數的乘積

取 $f=x^2$、$g=x^3$，$fg=x^5$。在 $x=1$，真正的導數是多少？$f'g'$ 又是多少？

In [ ]:
print("真正的 (fg)′ 在 x = 1：", round(d(lambda x: x**5, 1), 4))
print("錯誤的 f′·g′ 在 x = 1：", 2 * 1 * 3 * 1**2)
print("乘法法則 f′g + fg′    ：", 2 * 1 * 1**3 + 1**2 * 3 * 1**2)

## 3. ✏️ 動手 1：心輸出量

心輸出量 = 心跳 × 每搏輸出量：$CO(t)=HR(t)\cdot SV(t)$，其中 $HR=70+5t$（次/分）、$SV=70+t$（mL）。兩個量都在變，所以要用乘法法則：

$$CO'=HR'\cdot SV+HR\cdot SV'=5(70+t)+(70+5t)$$

請算出 $t=2$ 時的 $CO'$，單位是 mL/min 每分鐘。

In [ ]:
HR = lambda t: 70 + 5 * t
SV = lambda t: 70 + t
CO = lambda t: HR(t) * SV(t)

answer_1 = None      # ✏️ 填空：CO′(2) 是多少？可以用上面的公式寫算式

# ---- 自動檢查（不用改）----
print("t = 0 時電腦算出 CO′ ≈", round(d(CO, 0), 2), "（對照投影片的 420）")
if answer_1 is None:
    print("還沒填，把 t = 2 代入公式。")
else:
    check(abs(answer_1 - d(CO, 2)) < 0.01, "5(70 + t) + (70 + 5t) 代入 t = 2。")

## 4. 虛擬資料：從監測數據看變化率

真實的監測數據有雜訊。下面產生 20 分鐘的**虛擬**資料，用一條直線擬合 HR 和 SV，再算出斜率，就能估計它們的變化率。

In [ ]:
rng = np.random.default_rng(5)
t = np.arange(0, 21)
hr_obs = 70 + 5 * t + rng.normal(0, 3, t.size)
sv_obs = 70 + t + rng.normal(0, 1.5, t.size)

hr_slope = np.polyfit(t, hr_obs, 1)[0]
sv_slope = np.polyfit(t, sv_obs, 1)[0]
print(f"HR 的變化率估計 ≈ {hr_slope:.2f}（真值 5）")
print(f"SV 的變化率估計 ≈ {sv_slope:.2f}（真值 1）")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(t, hr_obs, "o", color="#028090"); ax[0].set_ylabel("HR (bpm)")
ax[1].plot(t, sv_obs, "o", color="#D9534F"); ax[1].set_ylabel("SV (mL)")
for a in ax:
    a.set_xlabel("t (minutes)"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. ✏️ 動手 2：酵素動力學與除法法則

Michaelis–Menten：$v(S)=\dfrac{V_{\max}S}{K_m+S}$，用除法法則得 $v'(S)=\dfrac{V_{\max}K_m}{(K_m+S)^2}$。取 $V_{\max}=100$、$K_m=5$。

導數的**正負**告訴你增減，**大小**告訴你增減有多快。

In [ ]:
v = lambda S: 100 * S / (5 + S)
vp = lambda S: 100 * 5 / (5 + S)**2

S = np.linspace(0, 40, 300)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(S, v(S), color="#028090", lw=3); ax[0].set_title("rate v(S)")
ax[1].plot(S, vp(S), color="#D9534F", lw=3); ax[1].set_title("v'(S)")
for a in ax:
    a.set_xlabel("S"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("S     v′ 公式    電腦算的")
for s in [0, 5, 20]:
    print(f"{s:<4}  {vp(s):<8}   {d(v, s):.3f}")

answer_2 = None      # ✏️ 填空：v′(15) 是多少？

# ---- 自動檢查（不用改）----
if answer_2 is None:
    print("還沒填，把 S = 15 代入公式。")
else:
    check(abs(answer_2 - 1.25) < 0.01, "500 / (5 + 15)²。")

## 6. 連鎖律：兩層變化率相乘

體表面積 $BSA(w)=\sqrt{170w/3600}$（身高 170 cm 不變），體重每月減少 2 kg。

- 外層：BSA 隨體重變，$dBSA/dw$
- 內層：體重隨時間變，$dw/dt=-2$

$$\frac{dBSA}{dt}=\frac{dBSA}{dw}\cdot\frac{dw}{dt}$$

## 7. ✏️ 動手 3：把兩層乘起來

In [ ]:
BSA = lambda w: (170 * w / 3600) ** 0.5

dBSA_dw = d(BSA, 80)      # 體重 80 kg 時，BSA 對體重的導數
dw_dt = -2                # 每月減少 2 kg
print(f"dBSA/dw ≈ {dBSA_dw:.4f} m²/kg")

dBSA_dt = None      # ✏️ 填空：用連鎖律，把上面兩個變數乘起來

# ---- 自動檢查（不用改）----
if dBSA_dt is None:
    print("還沒填，把 None 換成兩個變數相乘。")
else:
    check(abs(dBSA_dt - (-0.0242)) < 0.001, "是 dBSA_dw * dw_dt。")
    print(f"體表面積每月約減少 {abs(dBSA_dt):.3f} m²，所以化療劑量要定期重算。")

## 8. 連鎖律與 AI：梯度下降

神經網路就是一層層的合成函數，連鎖律負責算「誤差對每個權重的導數」。用課堂上最小的網路：

$$a=w_1x,\quad \hat y=w_2a,\quad L=(\hat y-y)^2$$

$x=2$、$y=1$，初始 $w_1=3$、$w_2=0.5$。連鎖律給出

$$\frac{\partial L}{\partial w_1}=2(\hat y-y)\,w_2\,x,\qquad \frac{\partial L}{\partial w_2}=2(\hat y-y)\,a$$

**✏️ 動手 4**：改 `lr`（學習率），例如 `0.001`、`0.03`、`0.2`，看誤差怎麼變。太大會發生什麼事？

In [ ]:
lr = 0.01     # ← 改這裡

x_, y_ = 2, 1
w1, w2 = 3.0, 0.5
history = []
for step in range(30):
    a = w1 * x_
    yhat = w2 * a
    L = (yhat - y_) ** 2
    history.append(L)
    if L > 1e6:
        print("誤差爆炸了：學習率太大，每一步都跨過頭。")
        break
    g1 = 2 * (yhat - y_) * w2 * x_
    g2 = 2 * (yhat - y_) * a
    w1, w2 = w1 - lr * g1, w2 - lr * g2

print("前兩步的誤差：", round(history[0], 2), "→", round(history[1], 2))
print("最後的誤差  ：", round(float(history[-1]), 6))

plt.figure(figsize=(7, 3.2))
plt.plot(history, "o-", color="#028090")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss L (log axis)"); plt.grid(alpha=0.3); plt.show()

## 9. 用程式檢查 AI 的答案

> AI 說：$BSA(w)=\sqrt{170w/3600}$ 的導數是 $\dfrac12\left(\dfrac{170w}{3600}\right)^{-1/2}$。

在 $w=80$ 比較一下。

In [ ]:
ai = lambda w: 0.5 * (170 * w / 3600) ** (-0.5)
print("AI 的答案在 w = 80：", round(ai(80), 4))
print("電腦算出的導數    ：", round(d(BSA, 80), 4))
print("兩者相差幾倍      ：", round(ai(80) / d(BSA, 80), 2), "   而 3600/170 =", round(3600 / 170, 2))

AI 漏乘了內層的導數 $170/3600$，所以差了 $3600/170$ 倍。連鎖律最常見的錯誤就是**忘記乘內層**。

## 小結

- 乘法、除法、連鎖律都可以用數值差分驗算。
- 多層的變化率要相乘，每一層都有單位。
- 神經網路的訓練，本質上就是連鎖律加上「沿導數反方向走一小步」。

---
*本 notebook 為教學練習，數值與情境皆為示意，資料為程式產生的虛擬資料，非真實病人資料，也不是用藥建議。*